# Results Overview

This notebook reads probe runs under `results/`, supports both the original classification runs and the newer regression runs, and visualizes layer-by-layer behavior.

In [9]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("ggplot")
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

In [10]:
ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "results"

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

Project root: /Users/tresi/Projects/AIMO-baseline
Results dir:  /Users/tresi/Projects/AIMO-baseline/results


In [11]:
def parse_run_path(path: Path) -> dict:
    rel = path.relative_to(RESULTS_DIR)
    if len(rel.parts) < 9:
        raise ValueError(f"Unexpected results path: {rel}")

    tail = rel.parts[-9:]
    prefix = rel.parts[:-9]
    probe_name, model_name, encoding, control_task, fold, seed, run_id, status, filename = tail

    layer_match = re.search(r"(?:_L|__L)(\d{3})", probe_name)
    if layer_match is None:
        raise ValueError(f"Could not parse layer from {probe_name}")

    return {
        "results_group": "/".join(prefix),
        "probe_name": probe_name,
        "layer": int(layer_match.group(1)),
        "model_name": model_name,
        "encoding": encoding,
        "control_task": control_task,
        "fold": int(fold),
        "seed": int(seed),
        "run_id": int(run_id),
        "status": status,
        "filename": filename,
        "path": path,
    }


def infer_metadata_path(results_group: str) -> Path | None:
    if "eval_adoption_absolute_accuracy_decay_aggregated" in results_group:
        candidate = ROOT / "data" / "eval_adoption_internals_aggregated" / "metadata.csv"
    elif "eval_adoption_absolute_accuracy_decay_table" in results_group:
        candidate = ROOT / "data" / "eval_adoption_internals_table" / "metadata.csv"
    else:
        candidate = ROOT / "data" / "internals" / "metadata.csv"
    return candidate if candidate.exists() else None


def last_non_null(series: pd.Series):
    cleaned = series.dropna()
    return cleaned.iloc[-1] if not cleaned.empty else np.nan


def maybe_metric(df: pd.DataFrame, column: str, reducer: str = "last"):
    if column not in df.columns:
        return np.nan
    series = df[column]
    if reducer == "last":
        return last_non_null(series)
    if reducer == "max":
        return series.max(skipna=True)
    if reducer == "min":
        return series.min(skipna=True)
    raise ValueError(reducer)


def load_metrics() -> tuple[pd.DataFrame, pd.DataFrame]:
    run_rows = []
    curve_frames = []

    for metrics_path in sorted(RESULTS_DIR.rglob("metrics.csv")):
        info = parse_run_path(metrics_path)
        df = pd.read_csv(metrics_path)

        curve = df.copy()
        for key, value in info.items():
            if key not in {"path", "filename"}:
                curve[key] = value
        curve_frames.append(curve)

        run_rows.append({
            **{k: v for k, v in info.items() if k not in {"path", "filename"}},
            "metrics_path": str(metrics_path),
            "epochs_logged": len(df),
            "best_val_loss": maybe_metric(df, "val loss", "min"),
            "best_val_acc": maybe_metric(df, "val acc", "max"),
            "best_val_f1": maybe_metric(df, "val f1", "max"),
            "best_val_error": maybe_metric(df, "val error", "min"),
            "best_val_pearson": maybe_metric(df, "val pearson", "max"),
            "full_test_acc": maybe_metric(df, "full test acc", "last"),
            "full_test_f1": maybe_metric(df, "full test f1", "last"),
            "unseen_test_acc": maybe_metric(df, "unseen test acc", "last"),
            "unseen_test_f1": maybe_metric(df, "unseen test f1", "last"),
            "full_test_error": maybe_metric(df, "full test error", "last"),
            "full_test_pearson": maybe_metric(df, "full test pearson", "last"),
            "unseen_test_error": maybe_metric(df, "unseen test error", "last"),
            "unseen_test_pearson": maybe_metric(df, "unseen test pearson", "last"),
        })

    runs_df = pd.DataFrame(run_rows).sort_values(["results_group", "layer", "control_task"]).reset_index(drop=True)
    curves_df = pd.concat(curve_frames, ignore_index=True).sort_values(["results_group", "layer", "control_task", "step"])
    return runs_df, curves_df


def load_predictions() -> pd.DataFrame:
    pred_frames = []

    for preds_path in sorted(RESULTS_DIR.rglob("preds.csv")):
        info = parse_run_path(preds_path)
        df = pd.read_csv(preds_path)
        df = df.rename(columns={df.columns[0]: "row_idx"})
        df["problem_id"] = df["instance"].str.extract(r"problem_(\d+)").astype(float).astype("Int64")
        df["row_id"] = df["instance"].str.extract(r"row_(\d+)").astype(float).astype("Int64")
        df["pred_abs_error"] = (df["pred"] - df["label"]).abs()
        df["pred_is_exact"] = np.isclose(df["pred"], df["label"]).astype(int)
        for key, value in info.items():
            if key not in {"path", "filename"}:
                df[key] = value

        metadata_path = infer_metadata_path(info["results_group"])
        if metadata_path is not None:
            metadata = pd.read_csv(metadata_path)
            merge_key = "row_id" if "row_id" in metadata.columns and df["row_id"].notna().any() else "problem_id"
            if merge_key in metadata.columns:
                df = df.merge(metadata, on=merge_key, how="left", suffixes=("", "_meta"))

        pred_frames.append(df)

    return pd.concat(pred_frames, ignore_index=True).sort_values(["results_group", "layer", "control_task"]).reset_index(drop=True)


runs_df, curves_df = load_metrics()
preds_df = load_predictions()

ValueError: You are trying to merge on Int64 and str columns for key 'problem_id'. If you wish to proceed you should use pd.concat

In [ ]:
summary = pd.DataFrame({
    "results_groups": [", ".join(sorted(runs_df["results_group"].unique()))],
    "n_metric_runs": [len(runs_df)],
    "n_prediction_rows": [len(preds_df)],
    "n_layers": [runs_df["layer"].nunique()],
    "controls": [", ".join(sorted(runs_df["control_task"].unique()))],
    "models": [", ".join(sorted(runs_df["model_name"].unique()))],
})
summary

In [ ]:
runs_df.head()

In [ ]:
classification_metrics = [
    "full_test_acc",
    "full_test_f1",
    "unseen_test_acc",
    "unseen_test_f1",
]
regression_metrics = [
    "full_test_error",
    "full_test_pearson",
    "unseen_test_error",
    "unseen_test_pearson",
]
available_classification = [c for c in classification_metrics if c in runs_df.columns and runs_df[c].notna().any()]
available_regression = [c for c in regression_metrics if c in runs_df.columns and runs_df[c].notna().any()]
metric_cols = available_classification if available_classification else available_regression
metric_cols

In [ ]:
control_summary = (
    runs_df.groupby(["results_group", "control_task"])[metric_cols]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)
control_summary

In [ ]:
top_metric = metric_cols[0]
ascending = top_metric.endswith("error")
top_runs = runs_df.sort_values(top_metric, ascending=ascending).head(10)
top_runs[["results_group", "layer", "control_task", *metric_cols]]

In [ ]:
ncols = min(2, len(metric_cols))
nrows = int(np.ceil(len(metric_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

plot_group = runs_df["results_group"].iloc[-1]
plot_df = runs_df[runs_df["results_group"] == plot_group]

for ax, metric in zip(axes, metric_cols):
    for control_task, group in plot_df.groupby("control_task"):
        group = group.sort_values("layer")
        ax.plot(group["layer"], group[metric], marker="o", linewidth=2, label=control_task)
    ax.set_title(f"{plot_group}: {metric.replace('_', ' ')}")
    ax.set_xlabel("Layer")
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_xticks(sorted(plot_df["layer"].unique()))

for ax in axes[len(metric_cols):]:
    ax.axis("off")

axes[0].legend(title="Control")
plt.tight_layout()

In [ ]:
heatmap_metric = metric_cols[0]
heatmap_df = plot_df.pivot(index="control_task", columns="layer", values=heatmap_metric).sort_index()

fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="viridis")
ax.set_title(f"{plot_group}: {heatmap_metric} by control and layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Control")
ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns)
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
plt.colorbar(im, ax=ax, label=heatmap_metric)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, plot_df["control_task"].nunique(), figsize=(4 * plot_df["control_task"].nunique(), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (control_task, group) in zip(axes, plot_df.groupby("control_task")):
    group = group.sort_values("layer")
    ax.plot(group["layer"], group["best_val_loss"], marker="o")
    ax.set_title(control_task)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Best val loss")
    ax.set_xticks(group["layer"])
plt.tight_layout()

In [ ]:
curve_subset = curves_df[(curves_df["results_group"] == plot_group) & (curves_df["layer"].isin([0, 6, 12, 18, 24]))].copy()
fig, axes = plt.subplots(1, curve_subset["control_task"].nunique(), figsize=(5 * curve_subset["control_task"].nunique(), 4), sharey=True)
axes = np.atleast_1d(axes)

for ax, control_task in zip(axes, sorted(curve_subset["control_task"].unique())):
    group = curve_subset[curve_subset["control_task"] == control_task]
    for layer, layer_df in group.groupby("layer"):
        ax.plot(layer_df["step"], layer_df["val loss"], marker="o", label=f"L{layer:03d}")
    ax.set_title(f"Validation loss: {control_task}")
    ax.set_xlabel("Step")
    ax.set_ylabel("Val loss")
axes[0].legend(title="Layer")
plt.tight_layout()

In [ ]:
prediction_summary = (
    preds_df.groupby(["results_group", "control_task", "layer", "seen"])
    .agg(
        n_examples=("pred_abs_error", "size"),
        exact_match_rate=("pred_is_exact", "mean"),
        avg_abs_error=("pred_abs_error", "mean"),
        avg_loss=("loss", "mean"),
    )
    .reset_index()
)
prediction_summary.head(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
pred_plot_df = prediction_summary[prediction_summary["results_group"] == plot_group]

for control_task, group in pred_plot_df[pred_plot_df["seen"] == "unseen"].groupby("control_task"):
    group = group.sort_values("layer")
    axes[0].plot(group["layer"], group["avg_abs_error"], marker="o", linewidth=2, label=control_task)
    axes[1].plot(group["layer"], group["avg_loss"], marker="o", linewidth=2, label=control_task)

axes[0].set_title("Average absolute error on unseen examples")
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Absolute error")
axes[1].set_title("Average loss on unseen examples")
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Loss")
axes[0].legend(title="Control")
plt.tight_layout()

In [ ]:
problem_cols = [
    c for c in [
        "results_group",
        "problem_id",
        "row_id",
        "original_problem",
        "model_is_robust",
        "absolute_accuracy_decay",
        "layer",
        "control_task",
        "pred",
        "label",
        "pred_abs_error",
        "loss",
        "seen",
    ] if c in preds_df.columns
]
preds_df[problem_cols].sort_values(["results_group", "control_task", "layer"]).head(20)